<img src="ebay.png" style="display:block; margin:auto" width=350>

<div align="center"> <h1 align="center"> Project: Used Car Listings on eBay Kleinanzeigen </h1> </div>

In this project we will analyze a [dataset](https://app.dataquest.io/c/54/m/294/guided-project%3A-exploring-ebay-car-sales-data/1/introduction?path=2&slug=data-scientist&version=2.3) comprising used cars sourced from eBay Kleinanzeigen, which is a classifieds segment on the German eBay website. On this platform, individuals buy, sell and trade a range of goods, including used cars. This section is specifically aimed at local communities and offers a diverse selection of used vehicles, from budget options to high-end models. Users can navigate an extensive selection, engage directly with sellers to negotiate offers, explore vehicle histories, and obtain important details to make informed purchasing decisions.

The dataset was initially gathered through web scraping and subsequently made available on Kaggle. The current dataset utilized in this analysis consists of 50,000 data points, curated by Dataquest. This version was prepared to simulate a less-refined representation of the data. The principal objective of this project revolves around data cleansing and the analyzation of the included used car listings.

The accompanying data dictionary outlining the dataset's attributes is provided below:

- `dateCrawled`: When this ad was first crawled. All field-values are taken from this date.
- `name`: Name of the car.
- `seller`: Whether the seller is private or a dealer.
- `offerType`: The type of listing
- `price`: The price on the ad to sell the car.
- `abtest`: Whether the listing is included in an A/B test.
- `vehicleType`: The vehicle Type.
- `yearOfRegistration`: The year in which the car was first registered.
- `gearbox`: The transmission type.
- `powerPS`: The power of the car in PS.
- `model`: The car model name.
- `kilometer`: How many kilometers the car has driven.
- `monthOfRegistration`: The month in which which year the car was first registered.
- `fuelType`: What type of fuel the car uses.
- `brand`: The brand of the car.
- `notRepairedDamage`: If the car has a damage which is not yet repaired.
- `dateCreated`: The date on which the eBay listing was created.
- `nrOfPictures`: The number of pictures in the ad.
- `postalCode`: The postal code for the location of the vehicle.
- `lastSeenOnline`: When the crawler saw this ad last online.
  
Lets start

*This project was completed as part of the Data Science Career Path offered by dataquest.io.*

#### The Data

In [ ]:
#import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
autos = pd.read_csv('autos.csv', encoding='Latin-1')

autos.info()

In [ ]:
autos.head()

We created a dataframe named 'autos' with the csv file 'autos.csv'. There are 15 string columns and 5 int columns in this dataframe. Furthermore, there is a total of 20 columns with ~50000 rows. Some columns have null values, but none have more than ~20% null values. The column names use camelcase instead of Python's preferred snakecase, which means we can't just replace spaces with underscores.

We'll first clean the column names to make the data easier to work with.

#### Clean Columns

In [ ]:
autos.columns

In [ ]:
columns = ['date_crawled', 'name', 'seller', 'offer_type', 'price', 'abtest',
       'vehicle_type', 'registration_year', 'gearbox', 'power_ps', 'model',
       'odometer', 'registration_month', 'fuel_type', 'brand',
       'unrepaired_damage', 'ad_created', 'nr_of_pictures', 'postal_code',
       'last_seen']

In [ ]:
autos.columns = columns

In the code above, we first looked at the columns names, then saved the columns to the variable columns and did some changes to the names. From camelcase to snakecase. Finally we reassigned the new column names to the dataframe.ns.

#### Data Exploration and Cleansing

We will start by conducting an exploratory analysis of the data to identify evident areas which require data cleansing.


In [ ]:
autos.describe(include='all') #show count, unique, top, freq, etc. of every column

In [ ]:
print(autos['seller'].value_counts())
print('\n')
print(autos['offer_type'].value_counts())

Our initial observations reveal following aspects:

Regarding the text columns, we observe, that all (or nearly all) of the values from the following columns are the same

- `'seller'`
- `'offer_type'`

We'll drop these columns due to there inbalance distribution.

In [ ]:
autos = autos.drop(["seller", "offer_type"], axis=1)

In [ ]:
print(autos['abtest'].value_counts())
print('\n')
print(autos['gearbox'].value_counts())

Above we used the `.describe(include='all')` method to show all the information about our columns. We then dived deeper into some columns, e.g. `'seller'` and `'offer_Type'` which are incredible inbalanced and therefore should be exluded from further analysis. Therefore we did drop these two columns. Other columns, like `'gearbox'` and `'abtest'` are also bivariate format, but with sensible distribution. 


In [ ]:
print(autos['price'].dtypes)
print(autos['odometer'].dtypes)

In [ ]:
autos['odometer'].unique()

As seen in the describe table above, the `'price'` and `'odometer'` columns look suspicious. While looking into it, The type seems to be `'object'` and not `'integer'`. The two columns contain numerical data with extra characters being stored as text. Our next step involves cleansing and converting these columns accordingly. 

In [ ]:
#transform odometer column to int and remove 'km' or ',' 
autos['odometer'] = autos['odometer'].str.replace('km', '').str.replace(',', '').astype(int) 

#rename the odometer column
autos.rename({'odometer' : 'odometer_km'}, axis=1, inplace=True)

autos['odometer_km'].head()

In [ ]:
#transform price column to int and remove '$' or ',' 
autos['price'] = autos['price'].str.replace('$', '').str.replace(',', '').astype(int)

autos['price'].head()

#### Diving Deeper Into Odometer and Price Column

Now we will explore more of the odometer_km and price columns and look for values that seem to be unrealistic.

In [ ]:
autos['odometer_km'].value_counts().sort_index(ascending=False)

The observations signify that the entries in this field include rounded values, implying that sellers likely selected from predetermined options for this particular attribute. Furthermore, there is a dominance of high kilometers vehicles compared to those with low kilometers.




In [ ]:
print(autos["price"].describe())
print(autos["price"].unique().shape)
autos["price"].value_counts().head(15)

As with the `'odometer'`, the `'prices'` appear to be rounded. However, considering the presence of 2357 different values in this column, this doesnt seem to be the right conclusion. Hence it might be possible, that sellers in general tend to round the values for the offer.

There are 1421 listings marked with a price of $0, which represents 2% of all cars. One consideration might be to remove these specific lines. Furthermore, the stated maximum price reaches an extraordinary sum of one hundred million dollars, prompting a closer look at the highest-priced entries.

In [ ]:
autos["price"].value_counts().sort_index(ascending=False).head(15)

In the preceding cells, adjustments were made to the dataset, primarily focusing on the `'odometer_km'` and `'price'` columns. While the `'odometer_km'` column doesn't have any outliers, the `'price'` column does. Consequently, in the subsequent step below, values exceeding 150,000 will be omitted.This value appears to be a reasonable threshold for the maximum price. Notably, values such as $0 were not excluded, considering the possibility that sellers might offer the car for free, considering the vehicle's condition.

In [ ]:
autos = autos[autos["price"].between(1,151000)]
autos["price"].describe()

#### The Date Columns

Several columns within the dataset encompass date-related information:

- `'date_crawled'`
- `'registration_month'`
- `'registration_year'`
- `'ad_created'`
- `'last_seen'`

These columns are made up of data that was crawled and data that contains meta information from the crawler. The non-registration data is stored as character strings.

Lets dive deeper into these columns and learn something about the data 

In [ ]:
autos[['date_crawled', 'ad_created', 'last_seen']][:5] #look at the frist five rows from the specific columns

We'll now look into the `'date_crawled'` column, we'll extract the date with `.str[:10]` and count the values. The output for every counted date will be in percentage due to `normalize=True`. We do not exclude the missing values, we then rank the values by date with `sort_index()` and by values with `sort_values()`.

##### Date Crawled

In [ ]:
#show the date plus value_counts per date, sorted by index value
autos['date_crawled'].str[:10].value_counts(normalize=True, dropna=False).sort_index(ascending=True) 

In [ ]:
#show the date plus value_counts per date, sorted by values ascending
autos['date_crawled'].str[:10].value_counts(normalize=True, dropna=False).sort_values()

The website seems to be got crawled daily for an approximate duration of one month spanning March and April 2016. As you can see below, the distribution of crawled listings across each day exhibits an approximately uniform pattern.

In [ ]:
plt.figure(figsize=(9, 4))
autos['date_crawled'].str[:10].value_counts(normalize=True, dropna=False).sort_index().plot(kind='line', rot=45, 
                                                                                            linewidth=0.8, color='lightcoral', alpha=0.8)
plt.xlabel('Dates', size=8)
plt.ylabel('Percentage Share per Date', size=8)
plt.title('Date Crawled', size=8)
plt.show()

##### Ad Created

In [ ]:
plt.figure(figsize=(9, 4))
autos['ad_created'].str[:10].value_counts(normalize=True, dropna=False).sort_index(ascending=True).plot(kind='line', rot=45, 
                                                                                                        linewidth=0.8, color='teal', alpha=0.8)
plt.xlabel('Dates', size=8)
plt.ylabel('Percentage Share per Date', size=8)
plt.title('Ad Created', size=8)
plt.show()

The range of ad created dates is quite various. While few of the created ads are quite old (with the oldest at around 9 months.), most of the ads fall within 1-2 months of the listing date. The maximum value is 0.0389. Hence 3.89% of the ads were created on 3th April of 2016.

In [ ]:
plt.figure(figsize=(9, 4))
autos['last_seen'].str[:10].value_counts(normalize=True, dropna=False).sort_index(ascending=True).plot(kind='line', rot=45, 
                                                                                                        linewidth=0.8, color='indigo', alpha=0.8)
plt.xlabel('Dates', size=8)
plt.xlabel('Dates', size=8)
plt.ylabel('Percentage Share per Date', size=8)
plt.title('Last Seen', size=8)
plt.show()

Since the crawler recorded the date it last saw any listing, it allows us to determine on what day a listing was removed, presumably signifying the sale of the respective car.

In the concluding three days the count of `last_seen` values increased disproportionately. Given that this figure is 6 to 10 times greater than the counts observed on preceding days, it is improbable that such a surge corresponds to a increase in car sales. It is more plausible that these elevated values are associated with the end of the crawling process. Hence these values are not indicative of actual car sales.

Next we will look deeper into the colum 'registration_year'.

#### Addressing Inaccuracies in Registration Year Data





In [ ]:
autos['registration_year'].describe()

The `registration_year` values have strange characteristics. The mean is given as 2005, with a remarkably high standard deviation of 105, suggesting the existence of data associated with 1900.

The range of `registration_year` spans from the minimum value of year 1000 (long before cars were invented) to the maximum value of year 9999 (long in the future). Taking into account the logical limitation that a car can't be first registered before the listing was seen, any car with a registration year after 2016 is inaccurate. Consequently, values above 2016 are excluded. 

Determining the earliest valid year is challenging. Practically, the range extends to the initial decades of the 1900s. One option is to remove the listings with these values. Let's answer the question: What percentage of our data in this column is invalid?

In [ ]:
# Show percentage of values with registration_year between 1900 and 2016
autos['registration_year'].between(1900, 2016).value_counts(normalize=True)

Roughly 96% of the cars are registered between 1900 - 2016. If we go further with the analysis, we would exclude about 4% of oberservations. Lets examine, how many values are above 2016.

In [ ]:
# Show percentage of values with registration_year between 2017 and 2023
autos['registration_year'].between(2017,2023).value_counts(normalize=True)

In [ ]:
# Exclude all values not between 1900 - 2016
autos = autos[autos["registration_year"].between(1900,2016)]

In [ ]:
# Examine the percentage distribution of registration year, show just first 10 values.
autos['registration_year'].value_counts(normalize=True).head(10)

As showen above, nearly all values are related to dates, which are above 2016. Dates beyond 2016 are inaccurate. Hence we can exclude these values a keep the range 1900 - 2016. Next we'll explore the price by brand.

#### Price by Brand

In [ ]:
# Show the brands listed
autos['brand'].unique()

In [ ]:
# Show the distribution by brand
autos['brand'].value_counts(normalize=True,).sort_values(ascending=False)

The german manufactures dominate the upper section of the rankings, collectively constituting nearly 50% of the comprehensive listings (Volkswagen ~21.4%, Mercedes-Benz ~9.4%, Audi ~ 8.5% and BMW ~10.8%). Just Porsche ist quite weak with ~0.5%, but that's probably because porsche is a very expensive premium brand and there aren't that many offers for these car.

Volkswagen stands out prominently as the preeminent brand. With about double the cars for sale of the next two brands combined. In contrast, numerous brands hold a less substantial share of the overall listings. Consequently, our analytical focus will be restricted to brands that account for more than 5% of the total listings.

In [ ]:
# Filter the brands by value_counts
brand_counts = autos["brand"].value_counts(normalize=True)
valuable_brands = brand_counts[brand_counts > .05].index

print(valuable_brands)

In [ ]:
mean_prices = {}

for brands in valuable_brands:
    brand = autos[autos["brand"] == brands]
    price_mean = brand["price"].mean()
    mean_prices[brands] = int(price_mean)

In [ ]:
mean_prices_sort = pd.Series(mean_prices).sort_values(ascending=False)
mean_prices_df = pd.DataFrame(mean_prices_sort, columns=['mean_price'])

Above, we computed the mean price for each brand, revealing a notable price disparity among the top brands. Upon scrutinizing the data, Mercedes-Benz, Audi, and BMW emerge as brands exhibiting the highest mean prices, while Opel and Ford showcase the lowest mean prices. Volkswagen, in contrast, maintains an intermediate mean price, a characteristic that may contribute to its popularity.ion.

In [ ]:
mean_kilometer = {}

for brands in valuable_brands:
    brand = autos[autos["brand"] == brands]
    kilometer_mean = brand["odometer_km"].mean()
    mean_kilometer[brands] = int(kilometer_mean)
    
mean_kilometer_sort = pd.Series(mean_kilometer).sort_values(ascending=False)
mean_price_kilometer = pd.DataFrame(mean_kilometer_sort, columns=['mean_kilometer'])
mean_price_kilometer['mean_price'] = mean_prices_df
mean_price_kilometer

In the above section we have expanded the analysis and also listed the average kilometers driven per brand. As you can see in the graph above, most of the brands have a average kilometer count from about 125000km - 130000km. The variation in kilometers among the top brands exhibits less disparity compared to the variation observed in prices. Specifically, all kilometers fall within a 10% range for the leading brands. 

Additionally, we can observe a trend. Wherein more expensive vehicles tend to feature higher kilometer, the less expensive counterparts tend to exhibit lower kilometers.

#### The Most Common Brand-Model Combinations

Next we are going to look into the most common brand-model combinations. Below we used the `.groupby()` operation, to complete these task.

In [ ]:
grouped = autos.groupby(['brand', 'model'])

# Show the 10 most common brand-model combinations
print(grouped.size().sort_values(ascending=False).head(10))

Based on this dataset, the most renowned brand-model pairing is represented by the Volkswagen Golf. Broadly speaking, Volkswagen boasts three distinct models within the top 10 listings. In comparison, BMW, Mercedes-Benz, and Opel each feature two models among the top 10, whereas Audi is represented by a singular model, namely the Audi A4.

#### Cars with and without damage

In [ ]:
# 33815 cars without damage, 4540 cars with damage
autos['unrepaired_damage'].value_counts()

We shall partition the `autos` dataset into two discrete subsets: one comprising vehicles with reported damage and another encompassing vehicles lacking any information regarding damage.

In [ ]:
damage = autos[autos['unrepaired_damage'] == 'ja']
no_damage = autos[autos['unrepaired_damage'] == 'nein']

Ultimately, we will compute the mean price for the two distinct categories and consolidate the results into a unified dataframe. As anticipated, vehicles with indications of damage manifest a markedly lower mean price when contrasted with those without such indications. Cars lacking damage exhibit an average price that is more than three times higher than their damaged counterparts.


In [ ]:
dmg_series = pd.Series([round(damage['price'].mean(),), round(no_damage['price'].mean(),)], index=['damage', 'no_damage'])
damage_df = pd.DataFrame(dmg_series, columns=['mean_price'])
damage_df

#### Conclusion

The primary aim of this project encompassed data cleaning and analysis of the provided used car listings. Following the completion of the data cleansing process, we processed the data and subsequently undertook a comprehensive dataset analysis. Based on the processed dataset, we identified the prominent brands, with four out of the six prominent brands originating from German manufacturers, and `Volkswagen` emerging as the most prevalent brand in the dataset. In terms of mean price, `Audi` exhibited the highest mean price value, while `Opel` demonstrated the lowest mean price value.

Upon delving into brand-model combinations, it is discerned that the most prevalent model is the `VW Golf`, followed by the `BMW 3 Series`. Other models such as the `VW Polo`, `Opel Corsa`, and `VW Passat`, while less frequent, still maintain a presence within the top 10.

Conclusively, an examination was conducted to compare the prices of listed cars with and without damage. Unsurprisingly, the findings indicated that cars without damage possessed higher market value. Due to our analysis within the scope of this dataset revealed that, on average, cars without damage are priced at over three times the value of their damaged counterparts.